# 📊 Tahap 2: Case Representation

**Tujuan:** Ekstraksi metadata, konten kunci, dan feature engineering dari teks Tahap 1.

**Output:** `SUBCPMK-3/processed/cases.csv`

---
| Langkah | Isi |
|---------|-----|
| **i** | Ekstraksi metadata -- nomor perkara, tanggal, jenis, pasal, pihak, hakim |
| **ii** | Ekstraksi konten kunci -- dakwaan, tuntutan, barang bukti, pertimbangan, amar |
| **iii** | Feature engineering -- word count, top terms, QA-pairs |
| **iv** | Simpan ke `cases.csv` |


## Cell 10 — Install Tambahan (Tahap 2)

In [ ]:
!pip install pandas scikit-learn -q
print('✅ pandas & scikit-learn siap!')


✅ pandas & scikit-learn siap!


## Cell 11 — Konfigurasi Path Output Tahap 2

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("/content/drive/MyDrive/SUBCPMK-3/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT = PROCESSED_DIR / "cases.csv"

print(f"📂 Input TXT     : {OUTPUT_DIR}")
print(f"📂 Processed dir : {PROCESSED_DIR}")
print(f"📄 Output CSV    : {CSV_OUT}")
txt_count = len(list(OUTPUT_DIR.glob('case_*.txt')))
print(f"   -> {txt_count} file .txt ditemukan")


📂 Input TXT     : /content/drive/MyDrive/SUBCPMK-3/raw
📂 Processed dir : /content/drive/MyDrive/SUBCPMK-3/processed
📄 Output CSV    : /content/drive/MyDrive/SUBCPMK-3/processed/cases.csv
   -> 50 file .txt ditemukan


## Cell 12 — Fungsi Ekstraksi Metadata & Konten Kunci

In [ ]:
import re

# ══════════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════════

def _clean_field(s):
    """
    Bersihkan sisa noise dari hasil tangkapan regex.
    Hapus: watermark, nomor halaman, disclaimer, kepaniteraan.
    Normalisasi: newline -> spasi, spasi ganda -> satu.
    """
    if not s:
        return ''
    # Hapus watermark 1 huruf di ujung
    s = re.sub(r'\s+[a-z]$', '', s.strip())
    # Hapus "hal. X dari Y ..." yang ikut tertangkap
    s = re.sub(r'\s*[a-z]?\s*hal\.?\s*\d+\s*dari\s*\d+[^\n]{0,100}', '', s, flags=re.IGNORECASE)
    # Hapus baris disclaimer / kepaniteraan / pelaksanaan
    s = re.sub(r'\n[^\n]*(?:disclaimer|kepaniteraan|pelaksanaan\s+fungsi)[^\n]*', '', s, flags=re.IGNORECASE)
    # Hapus "putusan nomor X hal. Y" inline
    s = re.sub(r'putusan\s+nomor\s+[\w/.]+\s+hal\.?\s*\d+[^\n]{0,80}', '', s, flags=re.IGNORECASE)
    # Newline -> spasi, normalisasi spasi ganda
    s = re.sub(r'\s*\n\s*', ' ', s)
    s = re.sub(r'\s{2,}', ' ', s)
    return s.strip()


# ══════════════════════════════════════════════════════════
# i. METADATA
# ══════════════════════════════════════════════════════════

def extract_nomor_perkara(text):
    """
    Tangkap: 106/Pid.Sus/2025/PN Pts
    Batas kanan ketat: newline atau spasi ganda.
    Max panjang 45 karakter.
    """
    m = re.search(
        r'nomor\s+'
        r'(\d+\s*/\s*[\w.]+\s*/\s*\d{4}\s*/\s*(?:pn|pa|pt|ms)\s*[\w]{1,15})'
        r'(?=\s*\n|\s{2,}|$)',
        text, re.IGNORECASE | re.MULTILINE
    )
    if m:
        val = re.sub(r'\s+', ' ', m.group(1)).strip().rstrip('.')
        if len(val) <= 45:
            return val
    # Fallback tanpa awalan "nomor"
    m2 = re.search(
        r'\b(\d+\s*/\s*pid[\w.]+\s*/\s*\d{4}\s*/\s*pn\s*[\w]{1,12})\b',
        text, re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', m2.group(1)).strip() if m2 else ''


def extract_tanggal_putusan(text):
    """
    Kembalikan format ISO YYYY-MM-DD.
    Pola: 'tanggal 12 Mei 2025' atau 'ditetapkan ... 12 Mei 2025'
    """
    BULAN = {
        'januari':'01','februari':'02','maret':'03','april':'04',
        'mei':'05','juni':'06','juli':'07','agustus':'08',
        'september':'09','oktober':'10','november':'11','desember':'12'
    }
    # Pola utama
    m = re.search(
        r'(?:pada\s+hari\s+\w+\s*[,.]?\s*)?'
        r'tanggal\s+(\d{1,2})\s+(\w+)\s+(\d{4})'
        r'(?!\s*/)',          # jangan tangkap angka dari nomor perkara
        text, re.IGNORECASE
    )
    if m:
        d, bln, y = m.group(1), m.group(2).lower(), m.group(3)
        b = BULAN.get(bln)
        if b:
            return f"{y}-{b}-{int(d):02d}"
    # Fallback: "ditetapkan di ... 12 Mei 2025"
    m2 = re.search(
        r'ditetapkan\b[^.\n]{0,80}?(\d{1,2})\s+(\w+)\s+(\d{4})',
        text, re.IGNORECASE
    )
    if m2:
        d, bln, y = m2.group(1), m2.group(2).lower(), m2.group(3)
        b = BULAN.get(bln)
        if b:
            return f"{y}-{b}-{int(d):02d}"
    return ''


def extract_jenis_perkara(text):
    t = text.lower()
    if re.search(r'/pid\.sus|tindak pidana khusus|narkotika|korupsi|tambang|kehutanan', t):
        return 'Pidana Khusus'
    if re.search(r'/pid\.b|perkara pidana biasa|tindak pidana umum', t):
        return 'Pidana Umum'
    if re.search(r'/pdt\.g|perkara perdata|gugatan', t):
        return 'Perdata Gugatan'
    if re.search(r'/pdt\.p|permohonan', t):
        return 'Perdata Permohonan'
    if re.search(r'/tun|tata usaha negara', t):
        return 'TUN'
    if re.search(r'peninjauan kembali', t):
        return 'Peninjauan Kembali'
    if re.search(r'terdakwa|dakwaan|penuntut umum', t):
        return 'Pidana Umum'
    return 'Lainnya'


def extract_pengadilan(text):
    m = re.search(
        r'(pengadilan\s+(?:negeri|tinggi|agama|militer|tata\s+usaha\s+negara)'
        r'\s+[\w\s]{2,25}?)'
        r'(?=\s+yang\s|\s+dalam\s|\s+kelas|\s*\n)',
        text, re.IGNORECASE
    )
    if m:
        # Buang kata ekor yang bukan nama kota
        val = re.sub(r'\s+', ' ', m.group(1)).strip().title()
        val = re.sub(r'\s+(Nomor|Yang|Dalam|Kelas|Berwenang|Mengadili).*$', '', val, flags=re.IGNORECASE)
        if 3 <= len(val) <= 60:
            return val
    return 'Mahkamah Agung' if re.search(r'mahkamah\s+agung', text, re.I) else ''


def extract_hakim_ketua(text):
    """
    Tangkap nama hakim ketua.
    FIX: teks sudah lowercase setelah clean_text(), jadi [A-Z] tidak match.
    Ganti dengan pola [a-z] dan tangkap sampai newline.
    """
    INVALID = {
        'menimbang','menyatakan','mengadili','bahwa','dengan','dalam',
        'didampingi','para','hakim','anggota','panitera','majelis',
        'dibantu','oleh','yang','telah','atas'
    }
    # Tangkap semua karakter hingga newline, lalu bersihkan
    patterns = [
        r'hakim\s+ketua\s+majelis\s*[:\-,]\s*([^\n]{3,60}?)(?=\n)',
        r'hakim\s+ketua\s*[:\-,]\s*([^\n]{3,60}?)(?=\n)',
        r'ketua\s+majelis\s*[:\-,]\s*([^\n]{3,60}?)(?=\n)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = _clean_field(m.group(1))
            # Buang gelar di akhir tapi pertahankan nama: "budi santoso, s.h., m.h." -> "Budi Santoso"
            val = re.sub(r',?\s*(?:s\.h|m\.h|s\.e|m\.kn|s\.ip|dr|prof)[\w.,\s]*$', '', val, flags=re.IGNORECASE).strip()
            words = set(val.lower().split())
            if val and 3 <= len(val) <= 60 and not words.intersection(INVALID):
                return val.title()
    return ''


def extract_pihak(text):
    """
    Kembalikan (p1_label, p1_value, p2_label, p2_value).

    Pihak1 (Terdakwa): tangkap hanya NAMA dari baris 'nama lengkap'.
    Batas tangkapan: berhenti di titik koma, newline, atau
    atribut identitas berikutnya (tempat, lahir, umur, dll).
    """
    # Kata-kata yang menandai batas akhir nama
    NAME_STOP = (
        r'(?=\s*[;\n]'
        r'|\s+(?:tempat|lahir|umur|tanggal|jenis|agama|alamat'
        r'|pekerjaan|pendidikan|kewarganegaraan|status|bin|binti'
        r'|als\b|alias\b))'
    )

    # ── Pidana: Terdakwa ─────────────────────────────────────
    terdakwa_hits = re.findall(
        r'nama\s+lengkap\s*[:\-]\s*'
        r'([^\n;]{3,80}?)'
        + NAME_STOP,
        text, re.IGNORECASE
    )
    if terdakwa_hits:
        # Ambil hanya bagian sebelum "als/alias/bin/binti" jika ada
        def trim_name(n):
            n = _clean_field(n)
            # Buang alias panjang: "Budi (bud) als Koko bin Saeful" -> "Budi"
            # Tapi pertahankan alias pendek dalam kurung
            n = re.sub(r'\s+(?:bin|binti)\s+.*$', '', n, flags=re.IGNORECASE)
            return n.strip()

        terdakwa = '; '.join(trim_name(t) for t in terdakwa_hits[:3])

        jpu_m = re.search(
            r'(?:jaksa\s+penuntut\s+umum|penuntut\s+umum)\s*[:\-,]?\s*'
            r'([^\n,;]{3,60}?)'
            r'(?=\s*[\n,;])',
            text, re.IGNORECASE
        )
        jpu = _clean_field(jpu_m.group(1)) if jpu_m else ''
        return 'Terdakwa', terdakwa, 'Penuntut Umum', jpu

    # ── Perdata: Penggugat & Tergugat ────────────────────────
    pg_m = re.search(
        r'(?:^|\n)\s*penggugat\s*[:\-,]?\s*'
        r'([^\n;]{3,80}?)'
        r'(?=\s*[;\n]|\s+(?:melawan|vs\b|lawan\b))',
        text, re.IGNORECASE
    )
    tg_m = re.search(
        r'(?:^|\n)\s*tergugat\s*[:\-,]?\s*'
        r'([^\n;]{3,80}?)'
        r'(?=\s*[;\n])',
        text, re.IGNORECASE
    )
    return (
        'Penggugat', _clean_field(pg_m.group(1)) if pg_m else '',
        'Tergugat',  _clean_field(tg_m.group(1)) if tg_m else '',
    )


def extract_pasal(text):
    """Kumpulkan pasal unik maks 10, gabung dengan ' | '."""
    hits = re.findall(
        r'pasal\s+\d+(?:\s+ayat\s+\(\d+\))?'
        r'(?:\s+(?:huruf\s+[a-z]|jo\.?\s+pasal\s+\d+))*',
        text, re.IGNORECASE
    )
    seen, out = set(), []
    for h in hits:
        key = re.sub(r'\s+', ' ', h.lower().strip())
        if key not in seen and len(out) < 10:
            seen.add(key)
            out.append(key)
    return ' | '.join(out)


# ══════════════════════════════════════════════════════════
# ii. KONTEN KUNCI
# ══════════════════════════════════════════════════════════

def _extract_block(text, start_pattern, stop_pattern, max_chars=600):
    """
    Ekstrak blok antara start_pattern dan stop_pattern.
    Buffer 4x max_chars agar stop_pattern punya ruang match.
    Lewatkan _clean_field sebelum dikembalikan.
    """
    m = re.search(start_pattern, text, re.IGNORECASE | re.DOTALL)
    if not m:
        return ''
    start_pos = m.end()
    rest = text[start_pos : start_pos + max_chars * 4]
    stop_m = re.search(stop_pattern, rest, re.IGNORECASE)
    block = rest[: stop_m.start()] if stop_m else rest
    return _clean_field(block)[:max_chars]


def extract_dakwaan(text):
    return _extract_block(
        text,
        r'(?:surat\s+)?dakwaan\s*(?:pertama|kedua|kesatu|primair|subsidiair)?\s*[:\n]',
        r'\n\n|tuntutan\s+pidana|menimbang\s+bahwa|fakta\s+hukum',
        max_chars=700
    )


def extract_tuntutan(text):
    """
    FIX: pola lama terlalu kaku -- hanya match jika ada spasi tepat sebelum [:].
    Ganti dengan pola fleksibel yang match semua varian format tuntutan.
    Contoh yang ditangkap:
      - 'tuntutan pidana:'
      - 'tuntutan jaksa penuntut umum adalah sebagai berikut:'
      - 'berdasarkan tuntutan penuntut umum yang pada pokoknya:'
    """
    return _extract_block(
        text,
        r'tuntutan\b[^:\n]{0,80}[:\n]',
        r'\n\n|pembelaan|pledoi|menimbang\s+bahwa|nota\s+pembelaan',
        max_chars=500
    )


def extract_barang_bukti(text):
    """
    FIX: stop pattern '\n\n' terlalu agresif -- di beberapa file barang bukti
    dipisah baris kosong antar item sehingga langsung berhenti di item pertama.
    Ganti dengan stop yang lebih kontekstual.
    """
    return _extract_block(
        text,
        r'barang\s+bukti\s*[:\n]',
        r'(?:menimbang|mempertimbangkan)\s*[,bahwa]'
        r'|keterangan\s+saksi'
        r'|fakta\s+(?:hukum|persidangan)'
        r'|telah\s+(?:didengar|membaca|memeriksa)'
        r'|m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i',
        max_chars=600
    )


def extract_pertimbangan_hukum(text):
    return _extract_block(
        text,
        r'(?:menimbang|mempertimbangkan)\s*,?\s*bahwa\s+',
        r'\n\n|m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i|amar\s+putusan|menyatakan\s+terdakwa',
        max_chars=700
    )


def extract_amar_putusan(text):
    return _extract_block(
        text,
        r'(?:m[\s]*e[\s]*n[\s]*g[\s]*a[\s]*d[\s]*i[\s]*l[\s]*i'
        r'|mengadili|amar\s+putusan)\s*[:\n]?',
        r'demikian\s+putusan|ditetapkan\s+di|hakim\s+ketua\s*[,:]'
        r'|panitera\s+pengganti|/\s*ttd',
        max_chars=900
    )


# ══════════════════════════════════════════════════════════
# iii. FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════

_STOPWORDS = {
    'yang','dan','di','ke','dari','dengan','untuk','dalam','adalah',
    'pada','oleh','ini','itu','tidak','atau','juga','akan','telah',
    'sebagai','bahwa','tersebut','kepada','para','serta','dapat',
    'ada','sudah','atas','bagi','agar','lebih','saat','hal','nomor',
    'pasal','ayat','undang','republik','indonesia','tentang','tahun',
    'negeri','pengadilan','mahkamah','agung','hakim','perkara',
    'putusan','setelah','sebelum','antara','terhadap','maka',
    'karena','hingga','terdakwa','saksi','kepada','masing',
    'jaksa','majelis','serta','halaman','direktori','disclaimer',
    'kepaniteraan','informasi','berusaha','mencantumkan','selalu',
}


def compute_word_count(text):
    return len(text.split())


def compute_top_terms(text, n=10):
    # Buang sisa header sebelum hitung top terms
    clean = re.sub(
        r'(?:disclaimer|kepaniteraan|direktori\s+putusan|halaman\s+\d+)[^\n]{0,200}',
        '', text, flags=re.IGNORECASE
    )
    tokens = re.findall(r'[a-z]{4,}', clean.lower())
    freq = {}
    for tok in tokens:
        if tok not in _STOPWORDS:
            freq[tok] = freq.get(tok, 0) + 1
    top = sorted(freq, key=lambda x: -freq[x])[:n]
    return ', '.join(top)


def make_qa_pairs(row):
    pairs = []
    if row.get('nomor_perkara'):
        pairs.append(f"Q: Apa nomor perkara ini? | A: {row['nomor_perkara']}")
    if row.get('tanggal_putusan'):
        pairs.append(f"Q: Kapan putusan dijatuhkan? | A: {row['tanggal_putusan']}")
    if row.get('jenis_perkara'):
        pairs.append(f"Q: Apa jenis perkara ini? | A: {row['jenis_perkara']}")
    if row.get('pengadilan'):
        pairs.append(f"Q: Di pengadilan mana perkara diputus? | A: {row['pengadilan']}")
    p1l = row.get('pihak1_label', '')
    p1  = row.get('pihak1', '')
    if p1l and p1:
        pairs.append(f"Q: Siapa {p1l}? | A: {p1[:150]}")
    if row.get('pasal'):
        pairs.append(f"Q: Pasal apa yang digunakan? | A: {row['pasal'][:200]}")
    if row.get('amar_putusan'):
        pairs.append(f"Q: Apa amar putusan hakim? | A: {row['amar_putusan'][:300]}")
    return ' || '.join(pairs)


print("✅ Semua fungsi Tahap 2 siap!")


✅ Semua fungsi Tahap 2 siap!


## Cell 13 — Proses Semua File TXT -> Ekstraksi Fitur

In [ ]:
import pandas as pd
from datetime import datetime

txt_files = sorted(OUTPUT_DIR.glob('case_*.txt'))

print('=' * 70)
print(f'Tahap 2 dimulai | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Total file      : {len(txt_files)}')
print('=' * 70)

rows = []

for i, txt_path in enumerate(txt_files, start=1):

    text = txt_path.read_text(encoding='utf-8')

    # ── i. Metadata ───────────────────────────────────────────────
    nomor        = extract_nomor_perkara(text)
    tanggal      = extract_tanggal_putusan(text)
    jenis        = extract_jenis_perkara(text)
    pengadilan   = extract_pengadilan(text)
    hakim        = extract_hakim_ketua(text)
    pasal        = extract_pasal(text)
    p1l, p1, p2l, p2 = extract_pihak(text)

    # ── ii. Konten kunci ──────────────────────────────────────────
    dakwaan      = extract_dakwaan(text)
    tuntutan     = extract_tuntutan(text)
    barang_bukti = extract_barang_bukti(text)
    pertimbangan = extract_pertimbangan_hukum(text)
    amar         = extract_amar_putusan(text)

    # ── iii. Feature engineering ──────────────────────────────────
    wc        = compute_word_count(text)
    top_terms = compute_top_terms(text)

    row = {
        'case_id'            : txt_path.stem,
        'source_file'        : txt_path.name,
        'nomor_perkara'      : nomor,
        'tanggal_putusan'    : tanggal,
        'jenis_perkara'      : jenis,
        'pengadilan'         : pengadilan,
        'hakim_ketua'        : hakim,
        'pihak1_label'       : p1l,
        'pihak1'             : p1,
        'pihak2_label'       : p2l,
        'pihak2'             : p2,
        'pasal'              : pasal,
        'dakwaan_ringkas'    : dakwaan,
        'tuntutan'           : tuntutan,
        'barang_bukti'       : barang_bukti,
        'pertimbangan_hukum' : pertimbangan,
        'amar_putusan'       : amar,
        'word_count'         : wc,
        'top_terms'          : top_terms,
    }
    row['qa_pairs'] = make_qa_pairs(row)
    rows.append(row)

    status = '✅' if nomor else '⚠️ '
    print(
        f"  [{i:02d}/{len(txt_files)}] {txt_path.name} | "
        f"{jenis:<18} | {wc:>6,} kata | "
        f"nomor: {nomor or '(tidak terdeteksi)'} {status}"
    )

print('=' * 70)
print(f'Selesai -- {len(rows)} baris siap diekspor')


2026-06-16 10:54:31,673 | INFO | NumExpr defaulting to 2 threads.
Tahap 2 dimulai | 2026-06-16 10:54:32
Total file      : 50
  [01/50] case_001.txt | Pidana Khusus      | 17,574 kata | nomor: 1001/pid.sus/2023/pn dps ✅
  [02/50] case_002.txt | Pidana Khusus      |  8,066 kata | nomor: 1024/pid.sus/2023/pn dps ✅
  [03/50] case_003.txt | Pidana Khusus      | 23,275 kata | nomor: 1027/pid.sus/2023/pn dps ✅
  [04/50] case_004.txt | Pidana Khusus      | 16,546 kata | nomor: 1031/pid.sus/2024/pn dps ✅
  [05/50] case_005.txt | Pidana Khusus      | 17,847 kata | nomor: 1055/pid.sus/2024/pn dps ✅
  [06/50] case_006.txt | Pidana Khusus      | 11,350 kata | nomor: 1066/pid.sus/2023/pn dps ✅
  [07/50] case_007.txt | Pidana Khusus      | 19,211 kata | nomor: 1073/pid.sus/2023/pn dps ✅
  [08/50] case_008.txt | Pidana Khusus      | 10,020 kata | nomor: 1076/pid.sus/2023/pn dps ✅
  [09/50] case_009.txt | Pidana Khusus      | 17,972 kata | nomor: 1078/pid.sus/2024/pn dps ✅
  [10/50] case_010.txt | Pida

## Cell 14 — Ekspor ke CSV

In [ ]:
import pandas as pd
import csv

df = pd.DataFrame(rows)
df = df.fillna('')

COL_ORDER = [
    'case_id', 'source_file',
    'nomor_perkara', 'tanggal_putusan', 'jenis_perkara',
    'pengadilan', 'hakim_ketua',
    'pihak1_label', 'pihak1', 'pihak2_label', 'pihak2',
    'pasal',
    'dakwaan_ringkas', 'tuntutan', 'barang_bukti',
    'pertimbangan_hukum', 'amar_putusan',
    'word_count', 'top_terms', 'qa_pairs',
]
cols  = [c for c in COL_ORDER if c in df.columns]
extra = [c for c in df.columns if c not in cols]
df    = df[cols + extra]

# ── Bersihkan newline dalam cell agar CSV tidak rusak ─────
for col in df.select_dtypes(include='object').columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('\n', ' ', regex=False)
        .str.replace('\r', ' ', regex=False)
        .str.strip()
    )

# ── Simpan CSV dengan quoting ketat ───────────────────────
#    quoting=QUOTE_ALL: semua field dibungkus " " sehingga
#    koma/titik koma di dalam nilai tidak merusak struktur
df.to_csv(
    CSV_OUT,
    index=False,
    encoding='utf-8-sig',
    quoting=csv.QUOTE_ALL,
)

print(f'✅ CSV tersimpan : {CSV_OUT}')
print(f'   Baris         : {len(df)}')
print(f'   Kolom         : {len(df.columns)}')
print(f'   Kolom detail  : {list(df.columns)}')


✅ CSV tersimpan : /content/drive/MyDrive/SUBCPMK-3/processed/cases.csv
   Baris         : 50
   Kolom         : 20
   Kolom detail  : ['case_id', 'source_file', 'nomor_perkara', 'tanggal_putusan', 'jenis_perkara', 'pengadilan', 'hakim_ketua', 'pihak1_label', 'pihak1', 'pihak2_label', 'pihak2', 'pasal', 'dakwaan_ringkas', 'tuntutan', 'barang_bukti', 'pertimbangan_hukum', 'amar_putusan', 'word_count', 'top_terms', 'qa_pairs']


## Cell 15 — Preview & Validasi Kelengkapan Kolom

In [ ]:
import pandas as pd

# ── Preview kolom ringkas ─────────────────────────────────
preview_cols = [
    'case_id', 'nomor_perkara', 'tanggal_putusan',
    'jenis_perkara', 'pengadilan', 'word_count'
]
preview_cols = [c for c in preview_cols if c in df.columns]
print(df[preview_cols].to_string(index=False))
print()

# ── Cek kelengkapan setiap kolom ─────────────────────────
print('─' * 65)
print('  Kelengkapan Kolom:')
print('─' * 65)
key_cols = [
    'nomor_perkara', 'tanggal_putusan', 'pengadilan', 'hakim_ketua',
    'pihak1', 'pihak2', 'pasal',
    'dakwaan_ringkas', 'tuntutan', 'barang_bukti',
    'pertimbangan_hukum', 'amar_putusan',
]
for col in key_cols:
    if col not in df.columns:
        continue
    filled = (df[col].astype(str).str.strip() != '').sum()
    pct    = round(filled / len(df) * 100)
    bar    = '█' * (pct // 5) + '░' * (20 - pct // 5)
    flag   = '' if pct >= 70 else ' ⚠️'
    print(f'  {col:<26} {filled:>2}/{len(df)} ({pct:>3}%) {bar}{flag}')
print()

# ── Sample satu kasus ─────────────────────────────────────
if len(df) > 0:
    r = df.iloc[0]
    print('─' * 65)
    print('  Sample case_001:')
    print(f'  Nomor    : {r["nomor_perkara"]}')
    print(f'  Tanggal  : {r["tanggal_putusan"]}')
    print(f'  Pihak1   : {str(r["pihak1"])[:80]}')
    print(f'  Amar     : {str(r["amar_putusan"])[:200]}')
    print(f'  QA[0]    : {str(r["qa_pairs"]).split("||")[0][:120]}')


 case_id            nomor_perkara tanggal_putusan jenis_perkara                  pengadilan  word_count
case_001 1001/pid.sus/2023/pn dps      2023-09-08 Pidana Khusus  Pengadilan Negeri Denpasar       17574
case_002 1024/pid.sus/2023/pn dps      2023-07-18 Pidana Khusus  Pengadilan Negeri Denpasar        8066
case_003 1027/pid.sus/2023/pn dps      2023-08-21 Pidana Khusus  Pengadilan Negeri Denpasar       23275
case_004 1031/pid.sus/2024/pn dps      2024-08-17 Pidana Khusus  Pengadilan Negeri Denpasar       16546
case_005 1055/pid.sus/2024/pn dps      2024-08-09 Pidana Khusus  Pengadilan Negeri Denpasar       17847
case_006 1066/pid.sus/2023/pn dps      2023-07-28 Pidana Khusus  Pengadilan Negeri Denpasar       11350
case_007 1073/pid.sus/2023/pn dps      2023-08-27 Pidana Khusus  Pengadilan Negeri Denpasar       19211
case_008 1076/pid.sus/2023/pn dps      2023-09-22 Pidana Khusus  Pengadilan Negeri Denpasar       10020
case_009 1078/pid.sus/2024/pn dps      2024-07-11 Pidana Khusus 

## Cell 16 — Statistik Ringkas

In [ ]:
print('=' * 60)
print('  STATISTIK TAHAP 2')
print('=' * 60)
print(f'  Total kasus     : {len(df)}')
print()

print('  Distribusi Jenis Perkara:')
for jenis, cnt in df['jenis_perkara'].value_counts().items():
    bar = '█' * cnt
    print(f'    {jenis:<22} {cnt:>2}  {bar}')
print()

print(f'  Word Count:')
print(f'    Min    : {int(df["word_count"].min()):>8,}')
print(f'    Max    : {int(df["word_count"].max()):>8,}')
print(f'    Rata2  : {int(df["word_count"].mean()):>8,}')
print(f'    Total  : {int(df["word_count"].sum()):>8,}')
print()
print('=' * 60)
print(f'  📄 Output: {CSV_OUT}')
print('=' * 60)


  STATISTIK TAHAP 2
  Total kasus     : 50

  Distribusi Jenis Perkara:
    Pidana Khusus          50  ██████████████████████████████████████████████████

  Word Count:
    Min    :    8,066
    Max    :   54,277
    Rata2  :   15,973
    Total  :  798,677

  📄 Output: /content/drive/MyDrive/SUBCPMK-3/processed/cases.csv


---
## Catatan Tahap 2

| Kolom | Deskripsi |
|-------|-----------|
| `nomor_perkara` | Nomor register perkara (contoh: `106/Pid.Sus/2025/PN Pts`) |
| `tanggal_putusan` | Format ISO `YYYY-MM-DD` |
| `jenis_perkara` | Pidana Khusus / Pidana Umum / Perdata Gugatan / TUN / Lainnya |
| `pengadilan` | Nama lengkap pengadilan |
| `hakim_ketua` | Nama hakim ketua majelis |
| `pihak1_label` / `pihak1` | Label & nama pihak pertama (Terdakwa atau Penggugat) |
| `pihak2_label` / `pihak2` | Label & nama pihak kedua (Penuntut Umum atau Tergugat) |
| `pasal` | Pasal-pasal yang dikutip, dipisah ` \| ` (maks 10) |
| `dakwaan_ringkas` | Ringkasan dakwaan / fakta (maks 700 karakter) |
| `tuntutan` | Tuntutan JPU (maks 500 karakter) |
| `barang_bukti` | Daftar barang bukti (maks 600 karakter) |
| `pertimbangan_hukum` | Pertimbangan hakim (maks 700 karakter) |
| `amar_putusan` | Amar/dispositif putusan (maks 900 karakter) |
| `word_count` | Jumlah kata dokumen |
| `top_terms` | Top 10 kata kunci (bag-of-words) |
| `qa_pairs` | QA-pairs otomatis, dipisah ` \|\| ` |

> Kolom dengan kelengkapan < 70% di Cell 15 berarti regex tidak cocok dengan format PDF tersebut.

> **Lanjut Tahap 3:** Data `cases.csv` ini siap digunakan untuk retrieval, similarity search, atau fine-tuning model.
